In [2]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [3]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/everest/main_cumulative_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Apply LRG mask
mask = cat['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# # Remove QSO targets
# mask = cat['DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# Require a minimum depth
min_depth = 800.
mask = cat['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# Julien's bad fibers list
bad_fibers = np.array(Table.read('/Users/rongpu/Documents/Data/desi_data/everest/misc/badfibers.csv')['FIBER'])
bad_fibers = np.append(bad_fibers, np.arange(2663, 2674+1))  # fibers affected by the CCD z5 defect
bad_fibers = np.append(bad_fibers, [3402, 3429])  # "swapped" fibers
bad_fibers = np.unique(bad_fibers)
print(len(bad_fibers), 'bad fibers')
mask_bad = np.in1d(cat['FIBER'], bad_fibers)
print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
cat = cat[~mask_bad]

print(len(cat))

FIBERSTATUS 340944 5119 0.014792104327824702
No data 340944 0 0.0
LRG mask 306665 34279 0.10054143789009339
Min depth 299348 7317 0.976140087717868
199 bad fibers
Bad fibers 287932 11416 0.038136216042866496
287932


In [4]:
# Custom DELTACHI2 vs z cut
d = (10**(3 - 3.5*cat['Z']))
mask_remove = (d>30) & (cat['DELTACHI2']<30)
mask_remove |= (d<30) & (cat['DELTACHI2']<d)
mask_remove |= (cat['DELTACHI2']<10)
mask_quality = cat['ZWARN']==0
mask_quality &= cat['Z']<1.4
mask_quality &= (~mask_remove)

print(np.sum(~mask_quality)/len(mask_quality))
cat = cat[mask_quality]
print(len(cat))

0.021793340094188905
281657


In [5]:
mask_star = (cat['SPECTYPE']=='STAR') | (cat['Z']<0.0003)
cat['SPECTYPE'][mask_star] = 'STAR'
print(np.sum(mask_star)/len(mask_star))

0.004583589259276354


In [6]:
t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'], return_counts=True)
t['frac (%)'] = t['count']/len(cat)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

SPECTYPE,count,frac (%)
str6,int64,float64
STAR,1291,0.5
QSO,5926,2.1
GALAXY,274440,97.4


In [7]:
# Exclude QSO targets
mask = cat['DESI_TARGET'] & 2**2==0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

277856 0.9865048622970493


SPECTYPE,count,frac (%)
str6,int64,float64
STAR,1210,0.4
QSO,3415,1.2
GALAXY,273231,98.3


In [8]:
# Select QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

3801 0.013495137702950752


SPECTYPE,count,frac (%)
str6,int64,float64
STAR,81,2.1
GALAXY,1209,31.8
QSO,2511,66.1


------
# Spectype of masked LRG targets

In [4]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/everest/main_cumulative_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Apply LRG mask
mask = cat['lrg_mask']!=0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# # Remove QSO targets
# mask = cat['DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# cat = cat[mask]

# Require a minimum depth
min_depth = 800.
mask = cat['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# Julien's bad fibers list
bad_fibers = np.array(Table.read('/Users/rongpu/Documents/Data/desi_data/everest/misc/badfibers.csv')['FIBER'])
bad_fibers = np.append(bad_fibers, np.arange(2663, 2674+1))  # fibers affected by the CCD z5 defect
bad_fibers = np.append(bad_fibers, [3402, 3429])  # "swapped" fibers
bad_fibers = np.unique(bad_fibers)
print(len(bad_fibers), 'bad fibers')
mask_bad = np.in1d(cat['FIBER'], bad_fibers)
print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
cat = cat[~mask_bad]

print(len(cat))

FIBERSTATUS 340944 5119 0.014792104327824702
No data 340944 0 0.0
LRG mask 34279 306665 0.8994585621099066
Min depth 33428 851 0.9751743049680562
199 bad fibers
Bad fibers 32153 1275 0.03814167763551514
32153


In [5]:
# Custom DELTACHI2 vs z cut
d = (10**(3 - 3.5*cat['Z']))
mask_remove = (d>30) & (cat['DELTACHI2']<30)
mask_remove |= (d<30) & (cat['DELTACHI2']<d)
mask_remove |= (cat['DELTACHI2']<10)
mask_quality = cat['ZWARN']==0
mask_quality &= cat['Z']<1.4
mask_quality &= (~mask_remove)

print(np.sum(~mask_quality)/len(mask_quality))
cat = cat[mask_quality]
print(len(cat))

0.033620501974932354
31072


In [6]:
mask_star = (cat['SPECTYPE']=='STAR') | (cat['Z']<0.0003)
cat['SPECTYPE'][mask_star] = 'STAR'
print(np.sum(mask_star)/len(mask_star))

0.10404866117404737


In [7]:
t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'], return_counts=True)
t['frac (%)'] = t['count']/len(cat)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

SPECTYPE,count,frac (%)
str6,int64,float64
QSO,652,2.1
STAR,3233,10.4
GALAXY,27187,87.5


In [8]:
# Exclude QSO targets
mask = cat['DESI_TARGET'] & 2**2==0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

30447 0.9798854273944387


SPECTYPE,count,frac (%)
str6,int64,float64
QSO,356,1.2
STAR,3067,10.1
GALAXY,27024,88.8


In [9]:
# Select QSO targets
mask = cat['DESI_TARGET'] & 2**2>0
print(np.sum(mask), np.sum(mask)/len(mask))

t = Table()
t['SPECTYPE'], t['count'] = np.unique(cat['SPECTYPE'][mask], return_counts=True)
t['frac (%)'] = t['count']/np.sum(mask)*100
t['frac (%)'].format = '%.1f'
t.sort('count')
t

625 0.020114572605561275


SPECTYPE,count,frac (%)
str6,int64,float64
GALAXY,163,26.1
STAR,166,26.6
QSO,296,47.4
